In [1]:
import pandas as pd
import sqlite3
import os

# Load your clean data
df = pd.read_csv('../data/processed/lending_club_clean.csv', low_memory=False)

# Create a SQLite database in your project folder
conn = sqlite3.connect('../data/processed/lending_club.db')

# Load the dataframe into a SQL table called 'loans'
df.to_sql('loans', conn, if_exists='replace', index=False)

print(f"Database created ✅")
print(f"Table 'loans' loaded with {len(df):,} rows")
print(f"Columns: {list(df.columns)}")

conn.close()

Database created ✅
Table 'loans' loaded with 765,050 rows
Columns: ['loan_amnt', 'funded_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'purpose', 'addr_state', 'dti', 'earliest_cr_line', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_pymnt', 'total_rec_prncp', 'total_rec_int', 'last_pymnt_d', 'recoveries']


In [2]:
# Reconnect and run a quick test query
conn = sqlite3.connect('../data/processed/lending_club.db')

test = pd.read_sql_query("SELECT * FROM loans LIMIT 5", conn)
print("Database connection working ✅")
print(f"\nFirst 5 rows:")
print(test.to_string())

conn.close()

Database connection working ✅

First 5 rows:
   loan_amnt  funded_amnt  term  int_rate  installment grade sub_grade  emp_length home_ownership  annual_inc verification_status     issue_d loan_status             purpose addr_state    dti earliest_cr_line  open_acc  pub_rec  revol_bal  revol_util  total_pymnt  total_rec_prncp  total_rec_int last_pymnt_d  recoveries
0     5000.0       5000.0    36     20.39       186.82     D        D4         8.0           RENT     50000.0            Verified  2018-03-01     Current               other         OK  21.80       2009-01-01       5.0      0.0      116.0        23.2  2043.690000          1219.69         824.00     Mar-2019         0.0
1    25000.0      25000.0    60     21.85       688.35     D        D5        10.0       MORTGAGE     65000.0     Source Verified  2018-03-01     Current  debt_consolidation         AL  12.89       1995-03-01       7.0      0.0     8657.0        98.4  7511.160000          2811.27        4699.89     Feb-2019     

In [1]:
# Run and display SQL file results
conn = sqlite3.connect('../data/processed/lending_club.db')

# Test Query — Default rate by grade
query = """
SELECT
    grade,
    COUNT(*) AS total_loans,
    ROUND(SUM(CASE WHEN loan_status = 'Defaulted'
        THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS default_rate_pct
FROM loans
GROUP BY grade
ORDER BY grade
"""

result = pd.read_sql_query(query, conn)
print("Default Rate by Grade:\n")
print(result.to_string())
conn.close()

NameError: name 'sqlite3' is not defined

In [2]:
import sqlite3
import pandas as pd

# Connect to database
conn = sqlite3.connect('../data/processed/lending_club.db')

# Default rate by grade
query = """
SELECT
    grade,
    COUNT(*) AS total_loans,
    ROUND(SUM(CASE WHEN loan_status = 'Defaulted'
        THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS default_rate_pct
FROM loans
GROUP BY grade
ORDER BY grade
"""

result = pd.read_sql_query(query, conn)
print("Default Rate by Grade:\n")
print(result.to_string())
conn.close()

Default Rate by Grade:

  grade  total_loans  default_rate_pct
0     A       167997              1.18
1     B       223546              3.41
2     C       224653              6.52
3     D       105505              8.99
4     E        32433             13.53
5     F         7658             21.91
6     G         3258             28.76


In [3]:
import sqlite3
import pandas as pd
import os

conn = sqlite3.connect('../data/processed/lending_club.db')

# Create excel data folder
os.makedirs('../excel', exist_ok=True)

# Table 1 — Loan status summary
q1 = """
SELECT
    loan_status,
    COUNT(*) AS total_loans,
    ROUND(SUM(loan_amnt), 2) AS total_value,
    ROUND(AVG(loan_amnt), 2) AS avg_loan_amount,
    ROUND(AVG(int_rate), 2) AS avg_interest_rate
FROM loans
GROUP BY loan_status
ORDER BY total_loans DESC
"""
df_status = pd.read_sql_query(q1, conn)
df_status.to_csv('../excel/summary_loan_status.csv', index=False)
print("✅ summary_loan_status.csv saved")

# Table 2 — Default rate by grade
q2 = """
SELECT
    grade,
    COUNT(*) AS total_loans,
    ROUND(AVG(loan_amnt), 2) AS avg_loan_amount,
    ROUND(AVG(int_rate), 2) AS avg_interest_rate,
    ROUND(SUM(CASE WHEN loan_status = 'Defaulted'
        THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS default_rate_pct
FROM loans
GROUP BY grade
ORDER BY grade
"""
df_grade = pd.read_sql_query(q2, conn)
df_grade.to_csv('../excel/summary_by_grade.csv', index=False)
print("✅ summary_by_grade.csv saved")

# Table 3 — Loan purpose summary
q3 = """
SELECT
    purpose,
    COUNT(*) AS total_loans,
    ROUND(SUM(loan_amnt), 2) AS total_value,
    ROUND(AVG(loan_amnt), 2) AS avg_loan_amount,
    ROUND(SUM(CASE WHEN loan_status = 'Defaulted'
        THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS default_rate_pct
FROM loans
GROUP BY purpose
ORDER BY total_loans DESC
"""
df_purpose = pd.read_sql_query(q3, conn)
df_purpose.to_csv('../excel/summary_by_purpose.csv', index=False)
print("✅ summary_by_purpose.csv saved")

# Table 4 — Top states
q4 = """
SELECT
    addr_state AS state,
    COUNT(*) AS total_loans,
    ROUND(SUM(loan_amnt), 2) AS total_value,
    ROUND(AVG(loan_amnt), 2) AS avg_loan_amount,
    ROUND(SUM(CASE WHEN loan_status = 'Defaulted'
        THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS default_rate_pct
FROM loans
GROUP BY state
ORDER BY total_loans DESC
LIMIT 20
"""
df_states = pd.read_sql_query(q4, conn)
df_states.to_csv('../excel/summary_by_state.csv', index=False)
print("✅ summary_by_state.csv saved")

# Table 5 — Monthly trend
q5 = """
SELECT
    SUBSTR(issue_d, 1, 7) AS month,
    COUNT(*) AS total_loans,
    ROUND(SUM(loan_amnt), 2) AS total_value,
    ROUND(AVG(loan_amnt), 2) AS avg_loan_amount
FROM loans
GROUP BY month
ORDER BY month
"""
df_monthly = pd.read_sql_query(q5, conn)
df_monthly.to_csv('../excel/summary_monthly.csv', index=False)
print("✅ summary_monthly.csv saved")

# Table 6 — Risk segmentation
q6 = """
SELECT
    CASE
        WHEN grade IN ('A', 'B') THEN 'Low Risk'
        WHEN grade IN ('C', 'D') THEN 'Medium Risk'
        WHEN grade IN ('E', 'F', 'G') THEN 'High Risk'
    END AS risk_category,
    COUNT(*) AS total_loans,
    ROUND(AVG(loan_amnt), 2) AS avg_loan_amount,
    ROUND(AVG(int_rate), 2) AS avg_interest_rate,
    ROUND(SUM(CASE WHEN loan_status = 'Defaulted'
        THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS default_rate_pct
FROM loans
GROUP BY risk_category
ORDER BY default_rate_pct
"""
df_risk = pd.read_sql_query(q6, conn)
df_risk.to_csv('../excel/summary_risk_segments.csv', index=False)
print("✅ summary_risk_segments.csv saved")

conn.close()
print("\nAll Excel data files exported ✅")
print(f"Location: excel/ folder")

✅ summary_loan_status.csv saved
✅ summary_by_grade.csv saved
✅ summary_by_purpose.csv saved
✅ summary_by_state.csv saved
✅ summary_monthly.csv saved
✅ summary_risk_segments.csv saved

All Excel data files exported ✅
Location: excel/ folder
